In [1]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# 2. 파일 업로드
from google.colab import files
import json

uploaded = files.upload()

Saving augmented_logic.json to augmented_logic.json
Saving logic.json to logic.json


In [3]:
# 3. 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# 4. 주요 라이브러리 로딩
import json
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback

In [5]:
# 5. 모델 및 토크나이저 로딩
model_name = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>', 'additional_special_tokens': ['<END>']})
model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(51201, 768)

In [6]:
# 6. Train Set 로딩
with open("augmented_logic.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

In [7]:
# 7. Validation Set 로딩
with open("logic.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

In [8]:
# 8. 정답생성용 프롬프트
def format_data(data):
    return [
        {
            "Text": f"문제: {item['Question']}\n정답: {item['Answer']}",
            "Question": f"문제: {item['Question']}\n",
            "Answer": f"정답: {item['Answer']}"
        }
        for item in data
    ]

formatted_train_data = format_data(train_data)
formatted_val_data = format_data(val_data)

In [9]:
# 9. 마스킹 토크나이즈 함수
def tokenize_with_mask(example):
    full_input = tokenizer(example["Text"], padding="max_length", truncation=True, max_length=512)
    question_input = tokenizer(example["Question"], padding="max_length", truncation=True, max_length=512)

    labels = full_input["input_ids"][:]
    q_len = sum([1 for id in question_input["input_ids"] if id != tokenizer.pad_token_id])

    # 질문 부분 마스킹
    labels[:q_len] = [-100] * q_len
    full_input["labels"] = labels

    return full_input

In [10]:
# 10. Dataset 변환 및 전처리
train_dataset = Dataset.from_list(formatted_train_data)
val_dataset = Dataset.from_list(formatted_val_data)

tokenized_train_dataset = train_dataset.map(tokenize_with_mask, batched=False)
tokenized_val_dataset = val_dataset.map(tokenize_with_mask, batched=False)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [11]:
# 11. Trainer 준비
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/KoGPT/CheckPoint/Logic/Answer",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=torch.cuda.is_available(),
    overwrite_output_dir=True
)

In [12]:
# 12. Trainer 구성
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

<ipython-input-12-da85c121d596>:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
# 13. 학습 실행
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,3.236100,2.869135
100,2.219700,1.877550
150,1.126100,1.573982
200,1.077600,1.053702
250,0.719600,0.996551
300,0.685900,0.807930
350,0.528600,0.764857
400,0.592000,0.650223
450,0.403000,0.601888
500,0.384000,0.545827


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1000, training_loss=0.7141603096723557, metrics={'train_runtime': 214.6341, 'train_samples_per_second': 9.318, 'train_steps_per_second': 4.659, 'total_flos': 522584064000000.0, 'train_loss': 0.7141603096723557, 'epoch': 10.0})

In [14]:
# 14. 모델 저장
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Answer")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Answer")

('/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Answer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Answer/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Logic/Answer/tokenizer.json')